# 07 · Generación real de los parques (XM)
**Dato tipo PUNTO** clave del trabajo: la generación horaria **real** que XM publica por recurso de generación.

1. Se descarga el listado de recursos de XM y se guardan los **solares**.
2. Cada parque de `parques_magdalena.csv` se busca por su palabra clave (`clave_xm`). **Revisa que la coincidencia sea correcta.**
3. Se descarga la generación horaria (métrica `Gene`, entidad `Recurso`) y se pasa a una fila por parque y hora, en **UTC** para poder cruzarla con GOES.

Librería: [`pydataxm`](https://github.com/EquipoAnaliticaXM/API_XM), el cliente oficial de la API de XM.

In [1]:
# ▶ CELDA DE PREPARACIÓN — ejecútala siempre de primera
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # En Colab los archivos se guardan en tu Google Drive para no perderlos
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/scm_magdalena")   # cámbiala si quieres
    CARPETA.mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA)
    PAQUETES = ["numpy", "scipy", "pandas", "xarray", "dask", "netCDF4", "h5netcdf", "s3fs", "pyproj", "shapely", "geopandas", "rasterio", "pvlib", "pydataxm", "libpysal", "esda", "matplotlib"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES], check=True)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())
assert Path("config.py").exists(), "No encuentro config.py: ejecuta primero el notebook 00 en esta misma carpeta"

Carpeta de trabajo: c:\Users\User\Desktop\Unal\Analisis Geoespacial\notebooks_scm_magdalena


In [2]:
import datetime as dt
import pandas as pd
import matplotlib.pyplot as plt
from pydataxm.pydataxm import ReadDB
import config as C

api = ReadDB()

def columna(df, *claves):
    for c in df.columns:
        if any(k in c.lower() for k in claves):
            return c
    raise KeyError(f"No encontré columna con {claves} en {list(df.columns)}")

## 1. Recursos solares registrados en XM

In [3]:
rec = api.request_data("ListadoRecursos", "Sistema", dt.date(2025, 1, 1), dt.date(2025, 1, 2))
rec.to_csv(C.RAW / "xm_listado_recursos.csv", index=False)
texto = rec.astype(str).apply(lambda f: " ".join(f).upper(), axis=1)
solares = rec[texto.str.contains("SOLAR")]
solares.to_csv(C.PROC / "xm_recursos_solares.csv", index=False)
print(f"Recursos solares en XM: {len(solares)}")
print("Columnas:", list(rec.columns))
solares.head(30)

Recursos solares en XM: 2214
Columnas: ['Id', 'Values_Code', 'Values_Name', 'Values_Type', 'Values_Disp', 'Values_RecType', 'Values_CompanyCode', 'Values_EnerSource', 'Values_OperStartdate', 'Values_State', 'Date']


,Id,Values_Code,Values_Name,Values_Type,Values_Disp,Values_RecType,Values_CompanyCode,Values_EnerSource,Values_OperStartdate,Values_State,Date
26,Sistema,2TYI,AUTOG CELSIA SOLAR YUMBO,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOGENERADOR,EPSG,RAD SOLAR,2017-09-03,PRUEBAS,2026-10-07
42,Sistema,2VJS,CELSIA SOLAR BOLIVAR,SOLAR,NO DESPACHADO CENTRALMENTE,NORMAL,EPSG,RAD SOLAR,2019-02-03,PRUEBAS,2026-10-07
48,Sistema,2YMA,AGPE NAFERTEX,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,ETTC,RAD SOLAR,2019-11-27,OPERACION,2026-10-07
50,Sistema,2YQM,AGPE TECNOEMPAQUES DE OCCIDENTE,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,BIAC,RAD SOLAR,2019-12-19,OPERACION,2026-10-07
52,Sistema,2Z66,AGPE SFV MCDONALDS SOLEDAD,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,GNCC,RAD SOLAR,2020-02-27,OPERACION,2026-10-07
54,Sistema,2ZHY,AUTOG CELSIA SOLAR ESPINAL,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOGENERADOR,EPSG,RAD SOLAR,2020-05-31,PRUEBAS,2026-10-07
55,Sistema,2ZJT,AGPE HOTEL HACIENDA CASONA DEL SALITRE,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,GNCC,RAD SOLAR,2020-05-21,OPERACION,2026-10-07
56,Sistema,2ZM3,AGPE DIEGO MONTESDEOCA ASSEFF,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,ETTC,RAD SOLAR,2020-06-11,OPERACION,2026-10-07
57,Sistema,2ZMH,AGPE DEVIMED MARINILLA,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,NRCC,RAD SOLAR,2020-06-18,OPERACION,2026-10-07
58,Sistema,2ZMJ,AGPE DEVIMED ENVIGADO,SOLAR,NO DESPACHADO CENTRALMENTE,AUTOG PEQ. ESCALA,NRCC,RAD SOLAR,2020-06-18,OPERACION,2026-10-07


## 2. Emparejar cada parque con su recurso en XM

In [4]:
c_cod = columna(rec, "code", "codigo")
c_nom = columna(rec, "name", "nombre")
parques = pd.read_csv(C.ARCHIVO_PARQUES)
mapa = {}
for _, p in parques.iterrows():
    m = solares[solares[c_nom].astype(str).str.upper().str.contains(str(p.clave_xm).upper())]
    if m.empty:
        print(f"✗ {p.nombre}: sin coincidencia para '{p.clave_xm}' (puede no reportar a XM)")
        continue
    print(f"✓ {p.nombre} → {m[[c_cod, c_nom]].values.tolist()}   ← verifica que sea el correcto")
    mapa[str(m.iloc[0][c_cod])] = p.nombre
mapa

✓ Parque Solar Fundación (Enel) → [['3HF5', 'FUNDACION'], ['3KR2', 'AGPE FUNDACION  OFTALMOLOGICA'], ['5E3E', 'AGPE FUNDACION CORAZON DE BELEN'], ['5FPX', 'AGPE FUNDACION PARA LA INTEGRACIÓN SOCIAL DE'], ['5MSD', 'AGPE OPORTUNIDAD DE CASINO ESPANA FUNDACION'], ['5UDP', 'AGPE FUNDACION CENTRO CULTURAL COLOMBO AMERIC']]   ← verifica que sea el correcto
✓ Caimán Cienaguero → [['3J4D', 'CAIMAN CIENAGUERO']]   ← verifica que sea el correcto


{'3HF5': 'Parque Solar Fundación (Enel)', '3J4D': 'Caimán Cienaguero'}

Si un parque quedó mal emparejado, corrige `mapa` a mano en la celda siguiente con el código correcto que viste en la tabla de solares, por ejemplo:
```python
mapa = {"XXXX": "Parque Solar Fundación (Enel)"}
```

In [5]:
# (opcional) corrección manual del emparejamiento
# mapa = {"CODIGO_XM": "Parque Solar Fundación (Enel)"}

## 3. Descargar la generación horaria del período
XM entrega **kWh por hora local**: `Values_Hour01` es de 00:00 a 01:00 hora de Colombia.

Cada intervalo se fecha al **inicio de la hora y en UTC** (+5 h).

In [8]:
gen = api.request_data("Gene", "Recurso", C.FECHA_INICIO.date(), C.FECHA_FIN.date())
g_cod = columna(gen, "code")
gen = gen[gen[g_cod].astype(str).isin(mapa)]
horas = [c for c in gen.columns if "hour" in c.lower()]
largo = gen.melt(id_vars=[g_cod, "Date"], value_vars=horas, var_name="h", value_name="kwh")
largo["h"] = largo["h"].str.extract(r"(\d+)", expand=False).astype(int)
local = pd.to_datetime(largo["Date"]) + pd.to_timedelta(largo["h"] - 1, unit="h")
largo["time"] = local + pd.Timedelta(hours=5)
largo["parque"] = largo[g_cod].astype(str).map(mapa)
largo["energia_mwh"] = pd.to_numeric(largo["kwh"], errors="coerce") / 1000
xm = largo[["parque", "time", "energia_mwh"]].sort_values(["parque", "time"])
xm.to_csv(C.PROC / "xm_generacion_horaria.csv", index=False)
print(len(xm), "filas guardadas")
xm.groupby("parque")["energia_mwh"].describe()

RuntimeError: Timeout context manager should be used inside a task

## 4. Revisión: ciclo diurno y días anómalos

In [7]:
xm["hora_local"] = (xm["time"] - pd.Timedelta(hours=5)).dt.hour
xm["dia"] = (xm["time"] - pd.Timedelta(hours=5)).dt.date
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
for parque, d in xm.groupby("parque"):
    d.groupby("hora_local")["energia_mwh"].mean().plot(ax=axs[0], marker="o", label=parque)
    d.groupby("dia")["energia_mwh"].sum().plot(ax=axs[1], label=parque)
axs[0].set_xlabel("Hora local"); axs[0].set_ylabel("MWh promedio"); axs[0].set_title("Ciclo diurno medio")
axs[1].set_ylabel("MWh/día"); axs[1].set_title("Energía diaria")
axs[0].legend(); plt.tight_layout(); plt.show()

NameError: name 'xm' is not defined

**Ojo:** los días con energía casi cero pueden deberse a mantenimiento o restricciones de red, no a nubes. Anótalos para excluirlos o discutirlos.

✅ Sigue con el notebook **08**.